# Sentralitet

Noen noder er alltid mer sentrale enn andre.  En nærmere undersøkelse viser at det er mange måter å "dfinere" det å være sentral, og alle har fordeler (og ulemper).
De sentrale (_no pun intended_) er 
- Populære noder (_degree centrality_)
- Bronoder (_betweenness centrality_)
- Sentrale noder (_closeness centrality_)
- Viktige noder (_page rank_)

Vi skal se på alle fire

## Hente inn to grafer som eksempler

### Epost-grafen

In [7]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np

In [8]:

import gzip

EG = nx.DiGraph()
# husk at gzip påpner i 'b'
with gzip.open("data/email.edgelist.txt.gz", "rt") as fd:
    for linje in fd:
         link = linje.split()
         EG.add_edge(int(link[0]), int(link[1]))
    #
#
print(f"Antall noder i grafen: {EG.number_of_nodes()}")
print(f"Antall kanter i grafen: {EG.number_of_edges()} (antall eposter)")
# Bort med eposter sendt til seg selv
EG.remove_edges_from(nx.selfloop_edges(EG))
print(f"Antall kanter etter selv-loop: {EG.number_of_edges()}")
# Og noder uten linker til seg nå
isolerte = list(nx.isolates(EG)) # kan ikke bruke iteratorer direkte
EG.remove_nodes_from(isolerte)
print(f"Antall noder etter isolerte: {EG.number_of_nodes()}")
# ANtall kanter i hver node
antall = []
for node, naboer in EG.degree():
    antall += [naboer]
    # eller antall = [d for n, d in EG.degree()]
#
print(f"Funnet Stdev:   {np.std(antall):.4f}")
print(f"Funnet Median:   {np.mean(antall):.4f}")

Antall noder i grafen: 57194
Antall kanter i grafen: 103731 (antall eposter)
Antall kanter etter selv-loop: 103083
Antall noder etter isolerte: 57189
Funnet Stdev:   36.0183
Funnet Median:   3.6050


lite sammenheng mellom median og standardavviket.
La oss se på noen tilfeldige noder:

In [12]:
import random

for i in range(10):
    node = int(random.random() * G.number_of_nodes())
    print(f"Node {node}: {len(G.in_edges(node))} {len(G.out_edges(node))}")

Node 17838: 1 1
Node 22214: 3 0
Node 27076: 1 0
Node 12013: 2 1
Node 33709: 1 1
Node 46048: 1 0
Node 44839: 0 2
Node 49156: 0 1
Node 48534: 1 1
Node 51345: 1 0


Men la oss se på noen utvalgte noder

In [11]:
print(f"63    ut: {len(G.out_edges(63))}  in: {len(G.in_edges(63))}")
print(f"40    ut: {len(G.out_edges(40))}   in: {len(G.in_edges(40))}")
print(f"407   ut: {len(G.out_edges(407))}   in: {len(G.in_edges(407))}")
print(f"1704  ut: {len(G.out_edges(1704))}  in: {len(G.in_edges(1704))}")
print(f"11798 ut: {len(G.out_edges(11798))} in: {len(G.in_edges(11798))}")
print(f"11028 ut: {len(G.out_edges(11028))} in: {len(G.in_edges(11028))}")
print(f"32199 ut: {len(G.out_edges(32199))} in: {len(G.in_edges(32199))}")

63    ut: 13  in: 29
40    ut: 0   in: 274
407   ut: 0   in: 205
1704  ut: 51  in: 107
11798 ut: 881 in: 262
11028 ut: 4170 in: 8
32199 ut: 6553 in: 2


### Les Misérables

In [9]:
with open("lesmiserables-character-network/parsed_data/jean-complete-node.csv", "r") as fd:
    noder_rå = fd.readlines()
#
# Bort med første linje
noder_rå = noder_rå[1:]

# Formatet er 
# "TH","Thénardier","Thénardier, innkeeper in Montfermeil, aka Jondrette"\n
# Hent ut det vi trenger, og lag noder

MG = nx.Graph()
for n in noder_rå:
    _s = n.split('"')
    MG.add_node(_s[1], Navn=_s[3], Rolle=_s[5])
#
print(f"Antall noder i grafen: {MG.number_of_nodes()}")

# Kantene
with open("lesmiserables-character-network/parsed_data/jean-complete-edge.csv", "r") as fd:
    kanter_rå = fd.readlines()
#
# Bort med første
kanter_rå = kanter_rå[1:]
print(f"Antall kanter: {len(kanter_rå)}")
#Formatet er 
# "MY","NP","Undirected","1","1.1.1" 
# hvor 1.1.1 er kapittelet hvor forbindelsen opptrer
for k in kanter_rå:
    _s = k.split(",")
    _ = MG.add_edge(_s[0][1:3], _s[1][1:3])
#
antall = []
for node, naboer in MG.degree():
    antall += [naboer]
    # eller antall = [d for n, d in EG.degree()]
#
print(f"Funnet Stdev:   {np.std(antall):.4f}")
print(f"Funnet Median:   {np.mean(antall):.4f}")

Antall noder i grafen: 181
Antall kanter: 1589
Funnet Stdev:   8.6738
Funnet Median:   5.4033


## Fire typer sentralitet


|Type|Hva som måles|Idé|Observasjon|
|---|---|---|---|
|Popularitet|Antall kanter|Hvor mange venner har du|Tar ikke hensyn til "kvaliteten" på vennene dine|
|Bronoder|Antall korteste (andres)<br>sti gjennom noden|Knytter sammen samfunnet,<br>mye informasjon flyter forbi|Du kan være en bro uten å være viktig selv|
|Sentrale|Antall korteste sti til andre noder|Nær der ting skjer|Følsom for endringer - tung å beregne|
|Viktig|Aggregert viktighet av naboene|Er viktig ved å kjenne mange viktige|Må iterere over hele grafen|



## Populære noder
Dette er det enkleste: Antall linker (til andre noder).  Kjører i $O(N)$ som en følge av at man må traversere alle noder (og vi husker at fordi sortering "bare" er $O(\log N)$ er totalen $O(N)$ og ikke $O(N) + O(N \log N)$).

Sier ikke så mye om hvorvidt noden er relevant eller ei.  Kan godt være at jeg kjenner flere i PIT enn Direktøren, men de hun kjenner er viktigere.

In [13]:
import time

start = time.time()
populæritet = nx.degree_centrality(EG)
top_populære_noder = sorted(populæritet, key=populæritet.get, reverse=True)[:5]
print("5 mest populære:")
for n in top_populære_noder:
    print(f"\t Node: {n} Naboer: {G.degree[n]}")
#
print(f"Å finne de fem mest populære tok {time.time()-start:.2f} sekunder")

5 mest populære:
	 Node: 32199 Naboer: 6555
	 Node: 11028 Naboer: 4178
	 Node: 13678 Naboer: 1228
	 Node: 11798 Naboer: 1143
	 Node: 13498 Naboer: 809
Å finne de fem mest populære tok 0.03 sekunder


Legg merke til at det her telles naboer uten å ta hensyn til retningen.

## Bronoder
En bro som knytter en øy til fastlandet, den er viktig fordi alle må over den.  Broen har bare to kanter til seg (én på hver side) og ikke populær (se over), og broen behøver ikke være nær noe viktig (sentrum av byen) men **posisjonen** relativt til andre noder i grafen gjør den viktig.  En bronode beskriver en strukturell plassering, heller enn egenskaper ved noden selv.

Teknisk er broen på korteste sti mellom mange andre (uansett hvem du skal besøke må du over broen).

Det følger av posisjonen at om en bronode feiler deles grafen.  Man finner bronoder ved å se hva som skjer når noder fjernes; de nodene som fører til at grafen separeres, de er bronoder.  Med andre ord: Ved å fjerne (fengsle?) bronoder fragmenterer man grafen.

Å finne bronoder krever å finne korteste sti for alle noder.  Raskeste algoritme er [Brande's algoritme](https://en.wikipedia.org/wiki/Brandes%27_algorithm) som kjører i $O(VE)$ tid.  Det betyr at kompleksiteten stiger lineært både med hvor "tett" den er og hvor "stor" den er.

Fordi bronoder står mellom grupper av andre norder er den engelske termen _betweenness_.

In [14]:
import time

start = time.time()
broer = nx.betweenness_centrality(MG)
top_bronoder = sorted(broer, key=broer.get, reverse=True)[:5]
print("5 viktigste bronoder")
for n in top_bronoder:
    print(f"\t Node: {n} Naboer: {MG.degree[n]}")
#
print(f"Å finne bronoder tok {time.time()-start:.2f} sekunder")


5 viktigste bronoder
	 Node: JV Naboer: 87
	 Node: FN Naboer: 25
	 Node: GA Naboer: 33
	 Node: MY Naboer: 18
	 Node: MA Naboer: 35
Å finne bronoder tok 0.04 sekunder


I en rettet graf må man være nøye med å definere hva en deling av grafen betyr (og derfor hva en bronode er).  Særlig: En node kan forbinde to deler i én retning men ikke den andre.

Legg merke til at vi ikke bruker epost-grafen som eksempel.  Å kjøre på epost-grafen tar tre kvarter, og resultatene er:

5 viktigste bronoder
-         Node: 11798 Naboer: 1143
-         Node: 1189 Naboer: 121
-         Node: 32199 Naboer: 6555
-         Node: 11028 Naboer: 4178
-         Node: 13498 Naboer: 809
Å finne bronoder tok 2873.13 sekunder

## Sentrale noder
Hvor tilgjengelig noden er for andre (lav gjennomsnitt korteste sti til andre noder).  Direktøren i PIT er trolig mest sentral i organisasjonen, som en følge av at hun har kortere vei "ned" til alle ansatte enn noen andre.

Sentrale noder har kort vei til andre, og vil derfor høre rykter hurtig.  Og, vice versa, kunne spre rykter og nyheter raskt.

Kompleksiteten aqvhenger ikke så mye av om grafen er rettet eller ei ($O(E+N)$ mot $O(E+N \log N$) for hver node, hvor $N$ er antall noder) men en rettet graf vil naturligvis ha "færre" kanten (fordi kanter i "andre retningen" ikke teller).  Videre kan en rettet graf være delt i betydningen av at én node kan nå til en annen, men ikke motsatt.

Om vi antar at kantene hverken har retning eller vekt, og har relativt få kanten (i forhold til maksimalt antall) da bruker vi bredde-først og kompleksiteten er $O(N^2)$. Om antall kanter er høyt nærmer kompleksiteten seg $O(N^3$) .  Det skal ikke så mye til før det er ønskelig med tilnærminger heller enn eksakte løsninger.

Det er en likhet mellom bronoder og sentrale noder.

Fordi kort vei til andre er essensen, heter dise nodene _closeness_ på engelsk.

In [17]:
import time

start = time.time()
UG = MG.to_undirected(MG) # Miserables-grafen
sentrale = nx.closeness_centrality(UG)
top_sentrale = sorted(sentrale, key=sentrale.get, reverse=True)[:5]
print("5 mest sentrale noder:")
for n in top_sentrale:
    print(f"\t Node: {n} Naboer: {UG.degree[n]}")
#   
print(f"Å finne sentrale noder {time.time()-start:.2f} sekunder")

5 mest sentrale noder:
	 Node: JV Naboer: 87
	 Node: JA Naboer: 29
	 Node: MA Naboer: 35
	 Node: CO Naboer: 25
	 Node: GA Naboer: 33
Å finne sentrale noder 0.02 sekunder


Det tok en halv time å kjøre på epost-grafen:

5 mest sentrale noder:
-          Node: 11028 Naboer: 4171
-          Node: 32199 Naboer: 6553
-          Node: 11798 Naboer: 1018
-          Node: 13498 Naboer: 802
-          Node: 12586 Naboer: 289
Å finne sentrale noder tok 1771.29 sekunder

## Viktige noder
Om vi myser litt, ser vi at populære, sentrale, og bronoder alle har sine egenskaper som en følge av sin plassering i grafen.  Altså ingen andre egenskaper enn posisjonen (lokasjonen).  I særdeleshet: det er ingen semantikk knyttet til rangeringene (bare "fysisk plassering").

Den mest kjente og brukte er *eigenvector centrality*.  For å si det forsiktig har Google gjort denne kjent under navnet **page rank**.  Essensen er at det er viktigere å ha viktige venner, enn mange venner.  Dersom mange sender epost til deg, da er du viktig.  De du sender epost til, blir viktige fordi du gjør det.  

I kontekst av en rettet graf kan vi tenke på det som en funksjon av hvor mange innkommende linker det er til en node (relativt til antall linker totalt), hvem de linkene kommer fra, og hvor mange utgående linker du har.  Jo flere utgående linker du har, jo flere "sprer" du din innflytese til og hvert av dine bidrag blir mindre.  

Om vi tenker på WWW så er det de sidene som mange linker til som er viktige, og sidene de viktige linker til, blir også viktige.  Om The New Times linker til din hjemmeside så blir din side interessant for mange.

Å finne hvem som er viktige er en iterativ prosess.  All "viktighet" fordeles likt over alle noder ($1/N$ på hver node).  For hver iterasjon fordeles hver nodes viktighet på alle noder den linker til.  Effekten blir at de nodene som har mange innkommende linker mottar mye, og alt de har fordeles til de nodene de linker til.  Etter "noen" iterasjoner vil situasjonen stabilisere seg.

En annen ekvivalent modell er å ha en "tilfeldig" surfer.  For hver node velges tilfeldig én av de utgående kantene.  Fra "tid til annen" velges et nytt startsted.  For hver iterasjon noteres det hvor surferen er.  Over tid vil antall ganger en node er besøkt representere hvor viktig noden er, eller med andre orde: Hva er sannsynligheten for at en node blir besøk.  Resultatet blir det samme som den andre itertive prosessen.

Fordi resultatet blir en vektor $v$ som er $N$ elementer lang.  Om vi setter opp nodene som en todimensjonal matrise $M$ hvor et element $n,m$ er satt om det er en link fra $n$ til $m$.  


kompleksiteten er "bare" $O(V+E)$.

In [23]:
# Beregne de viktigste nodene
page_rank = nx.eigenvector_centrality(MG)
viktigste_noder = sorted(page_rank, key=page_rank.get, reverse=True)[:5]
print("5 viktigste i Miserables:")
for n in viktigste_noder:
    print(f"\t Node {n}  har {MG.degree[n]} naboer")
#   

5 viktigste i Miserables:
	 Node JV  har 87 naboer
	 Node MA  har 35 naboer
	 Node GA  har 33 naboer
	 Node JA  har 29 naboer
	 Node CO  har 25 naboer


In [22]:
# Beregne de viktigste nodene
page_rank = nx.eigenvector_centrality(EG)
viktigste_noder = sorted(page_rank, key=page_rank.get, reverse=True)[:5]
print("5 viktigste i epost:")
for n in viktigste_noder:
    print(f"\t Node {n}  har {EG.degree[n]} naboer")
#   

5 viktigste i epost:
	 Node 11798  har 1143 naboer
	 Node 12586  har 333 naboer
	 Node 14603  har 297 naboer
	 Node 288  har 108 naboer
	 Node 33  har 112 naboer
